# Atlas Industries — Retrieval Quality Exploration

Validates the FAISS vector index against a sample of the gold eval cases.
Confirms that English queries, Arabic queries, and cross-language queries
all return relevant chunks with correct domain and source metadata.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.settings import settings
from src.ingest import build_embeddings, load_index

store_path = Path(settings.vector_store_path)
print(f"Vector store : {store_path}")
print(f"Exists       : {store_path.exists()}")
print(f"Embedding    : {settings.embedding_model}")
print(f"Top-k        : {settings.top_k}")

In [ ]:
embeddings = build_embeddings()
index = load_index(store_path, embeddings)
print(f"Index loaded. Total vectors: {index.index.ntotal}")

## Helper

Prints retrieved chunks with their metadata for quick inspection.

In [ ]:
def retrieve(query: str, k: int = settings.top_k, domain_filter: str | None = None):
    """Run a similarity search and pretty-print results."""
    if domain_filter:
        results = index.similarity_search(
            query, k=k, filter={"domain": domain_filter}
        )
    else:
        results = index.similarity_search(query, k=k)

    print(f"Query : {query}")
    print(f"Filter: {domain_filter or 'none'}")
    print("-" * 70)
    for i, doc in enumerate(results, 1):
        m = doc.metadata
        print(f"[{i}] {m['source_filename']}  |  domain={m['domain']}  |  lang={m['language']}")
        print(f"    {doc.page_content[:200].strip()}...")
        print()
    return results

## English Queries

In [ ]:
# EVAL-001 — HR, English
# Expected source: HR-001-vacation-policy.md
retrieve("How many vacation days does a new hire get in their first year?")

In [ ]:
# EVAL-004 — IT, English
# Expected source: IT-001-vpn-setup-guide.md
retrieve("What server and port do I use to connect to the company VPN?")

In [ ]:
# EVAL-007 — Finance, English, multi-source
# Expected sources: FIN-001-travel-reimbursement-policy.md, FIN-002-expense-claim-sop.pdf
retrieve("What is the domestic hotel cap and how many days to submit a travel claim?")

## Arabic Queries

In [ ]:
# EVAL-002 — HR, Arabic
# Expected source: HR-007-سياسة-الإجازات-المرضية.md
retrieve("ما هو عدد أيام الإجازة المرضية المدفوعة سنويًا، ومتى يلزم تقديم شهادة طبية؟")

In [ ]:
# EVAL-009 — Finance, Arabic
# Expected source: FIN-009-عشاء-العملاء.docx
retrieve("كم الحد الأقصى لتكلفة عشاء العملاء لكل شخص؟")

## Cross-Language Retrieval

EVAL-008 is an English question whose answer spans an English doc and an Arabic doc.
The index must surface the Arabic source even though the query is in English.

In [ ]:
# EVAL-008 — Finance, English query → expects FIN-007-سياسة-السفر-الدولي.md (Arabic)
results = retrieve(
    "What is the international per diem rate and does it cover client dinners?",
    k=6,
)

found_arabic_source = any(
    "FIN-007" in doc.metadata["source_filename"] for doc in results
)
print(f"Arabic source retrieved: {found_arabic_source}")

## Domain-Filtered Retrieval

Confirms that the `domain` metadata field can be used to scope searches.

In [ ]:
# Same query scoped to Finance only — should exclude HR and IT results
retrieve("How do I submit an expense claim?", domain_filter="finance")

## Index Statistics

In [ ]:
from collections import Counter

all_docs = index.docstore._dict.values()
domains = Counter(d.metadata["domain"] for d in all_docs)
languages = Counter(d.metadata["language"] for d in all_docs)
sources = len({d.metadata["source_filename"] for d in all_docs})

print(f"Total chunks : {index.index.ntotal}")
print(f"Unique sources: {sources}")
print(f"By domain    : {dict(domains)}")
print(f"By language  : {dict(languages)}")